In [ ]:
# ============================================================
#                     JAX SETUP
# ============================================================
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import jax
import jax.numpy as jnp
from jax import vmap
jax.config.update('jax_enable_x64', True)

print('Local devices:', jax.local_device_count(), flush=True)
print('Total devices:', jax.device_count(), flush=True)

In [ ]:
# ============================================================
#                     LIBRARIES
# ============================================================
import copy
import time
import pathlib
import numpy as np
import yaml
from deepmerge import always_merger
from scipy.integrate import trapezoid as scipy_trap
import matplotlib.pyplot as pl
import matplotlib.gridspec as gridspec

In [ ]:
# ============================================================
#                     GODMAX IMPORTS
# ============================================================
import sys
curr_path        = pathlib.Path().absolute()
abs_path_src     = os.path.abspath(curr_path / '../src/')
abs_path_params  = os.path.abspath(curr_path / '../param_files/')
abs_path_data    = os.path.abspath(curr_path / '../data/')
abs_path_results = os.path.abspath(curr_path / '../results/')
sys.path.append(os.path.join(str(abs_path_src), 'arxiv'))

from godmax.get_Cls  import get_Cl
from godmax.get_covs import get_cov
from godmax.base_class import get_vmapped_func, get_vmapped_func_warg

print('Imports done', flush=True)

In [ ]:
# ============================================================
#                     SETUP (params, ell, n(z))
# ============================================================
PARAM_DIR  = pathlib.Path(abs_path_params)
DATA_DIR   = pathlib.Path(abs_path_data)
OUTPUT_DIR = pathlib.Path(abs_path_results)
DNDZ_DIR   = pathlib.Path('/home/vtinnane/Documents/Codes/develop/tSZxEuclid/Data/dndz')
os.makedirs(OUTPUT_DIR / 'tests', exist_ok=True)
os.makedirs(OUTPUT_DIR / 'tests/mock-data', exist_ok=True)

def read_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

default_data    = read_yaml(PARAM_DIR / 'params_default.yaml')
experiment_data = read_yaml(PARAM_DIR / 'DESxACT/params_v0.yaml')
merged          = always_merger.merge(default_data, experiment_data)

sim_params_dict   = merged.get('sim_params',   {})
halo_params_dict  = merged.get('halo_params',  {})
analysis_dict     = merged.get('analysis',     {})
other_params_dict = merged.get('other_params', {})

# Ell array
lmin, lmax, dl_log = 10.0, 10000.0, 0.23025851
l_edges  = np.exp(np.arange(np.log(lmin), np.log(lmax), dl_log))
dl_array = jnp.array(l_edges[1:] - l_edges[:-1])
l_eff    = jnp.array((l_edges[1:] + l_edges[:-1]) / 2.0)
halo_params_dict['ell_array']    = l_eff
analysis_dict['l_array_survey']  = l_eff
analysis_dict['dl_array_survey'] = dl_array
nell = len(l_eff)

# Euclid source n(z) — 6 shear bins, normalised forecast
# Files: nz_norm_euclid_forecast_bin-{1..6}.txt  (col0=z, col1=n(z), integral=1)
nbins_source = 6
_nz_files = [np.loadtxt(DNDZ_DIR / f'nz_norm_euclid_forecast_bin-{i}.txt')
             for i in range(1, nbins_source + 1)]
z_src = _nz_files[0][:, 0]                          # common z grid (3000 pts, 0–6)
nz_src = {ji: np.maximum(_nz_files[ji][:, 1], 1e-4)
           for ji in range(nbins_source)}

nz_source_info = {'nbins': nbins_source, 'z_array_source': z_src}
for ji in range(nbins_source):
    nz_source_info[f'nz{ji}'] = nz_src[ji]
analysis_dict['nz_source_info_dict'] = nz_source_info
other_params_dict['Delta_z_bias_array']    = np.zeros(nbins_source)
other_params_dict['mult_shear_bias_array'] = np.zeros(nbins_source)

# Euclid lens n(z) — use same 6 bins
nbins_lens   = 6
nz_lens_info = {'nbins_lens': nbins_lens, 'z_array_lens': z_src.copy()}
for ji in range(nbins_lens):
    nz_lens_info[f'nz{ji}'] = nz_src[ji]
analysis_dict['nz_lens_info_dict'] = nz_lens_info

# Null yy noise
yy_noise_file = OUTPUT_DIR / 'tests/mock-data/yy-false-Cl.txt'
np.savetxt(yy_noise_file, np.column_stack([np.array(l_eff), 1e-25 * np.ones(nell)]))
analysis_dict['yy_noise_ell_fname'] = str(yy_noise_file)

# Covariance settings
analysis_dict['stats_for_cov']         = ['yy']
analysis_dict['fsky_yy']               = 0.1
analysis_dict['fsky_ky']               = 0.1
analysis_dict['fsky_kk']               = 0.1
analysis_dict['fsky_kg']               = 0.1
analysis_dict['fsky_gg']               = 0.1
analysis_dict['fsky_yg']               = 0.1
analysis_dict['sigma_epsilon_SN_bins'] = jnp.ones(nbins_source) * 0.26
analysis_dict['neff_arcmin2_SN_bins']  = jnp.ones(nbins_source) * 30.0   # Euclid ~30 gal/arcmin²
analysis_dict['nbar_lens_bins']        = jnp.ones(nbins_lens) * 10.0

print('Setup done', flush=True)
print(f'  nbins_source={nbins_source}, nbins_lens={nbins_lens}')
print(f'  z_src range: {z_src[0]:.3f} – {z_src[-1]:.3f}  (n={len(z_src)} pts)')
for ji in range(nbins_source):
    peak_z = z_src[np.argmax(nz_src[ji])]
    print(f'  source bin {ji}: peak_z={peak_z:.3f}')

In [ ]:
# ============================================================
#          INSTANTIATE OBJECTS  (get_cov is slow)
# ============================================================
print('Computing get_Cl...', flush=True)
t0 = time.time()
get_cl = get_Cl(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict)
print(f'  done in {time.time()-t0:.1f}s', flush=True)

print('\nComputing get_cov (slow — builds full trispectrum for yy)...', flush=True)
t0 = time.time()
get_cov_obj = get_cov(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict,
                      Cl_obj=get_cl)
print(f'  done in {time.time()-t0:.1f}s', flush=True)

In [ ]:
# ============================================================
#  TEST 1.1 — y3d_mat: positivity and magnitude
# ============================================================
# y3d_mat has shape (nr, nz, nM)
# Must be strictly positive; central value ~1e-6 to 1e-4 Mpc^-1 for M~1e14 at z~0.5

y3d = np.array(get_cl.y3d_mat)   # (nr, nz, nM)
r   = np.array(get_cl.r_array)
z   = np.array(get_cl.z_array)
M   = np.array(get_cl.M_array)

n_neg    = np.sum(y3d < 0)
n_total  = y3d.size

# Slice at M~1e14, z~0.5
jz_ref = np.argmin(np.abs(z - 0.5))
jM_ref = np.argmin(np.abs(np.log10(M) - 14.0))
y3d_slice = y3d[:, jz_ref, jM_ref]

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

axes[0].loglog(r, y3d_slice, 'b-', lw=2)
axes[0].axhspan(1e-6, 1e-4, alpha=0.15, color='green', label='expected range at centre')
axes[0].set_xlabel(r'$r$ [Mpc/h]', fontsize=12)
axes[0].set_ylabel(r'$y_{3D}(r)$ [Mpc$^{-1}$]', fontsize=12)
axes[0].set_title(f'y3d profile — z={z[jz_ref]:.2f}, log10(M)={np.log10(M[jM_ref]):.1f}', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

im = axes[1].pcolormesh(np.log10(M), z, np.log10(np.clip(y3d[0], 1e-30, None)),
                        cmap='plasma')
pl.colorbar(im, ax=axes[1], label=r'$\log_{10}(y_{3D}[r_{\rm min}])$')
axes[1].set_xlabel(r'$\log_{10}(M\,[M_\odot/h])$', fontsize=12)
axes[1].set_ylabel('z', fontsize=12)
axes[1].set_title('y3D at smallest r — all (z, M)', fontsize=11)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test1_1_y3d.png', dpi=100, bbox_inches='tight')
pl.show()

print(f'y3D @ r_min, M=1e14, z=0.5 : {y3d_slice[0]:.3e} Mpc^-1')
print(f'Negative values: {n_neg}/{n_total} ({100*n_neg/n_total:.3f}%)')
assert n_neg == 0, f'FAIL: {n_neg} negative y3d values'
print('PASS: y3d_mat is strictly positive everywhere')

In [ ]:
# ============================================================
#  TEST 1.2 — Tinker 2010 bias consistency integral
#  ∫ (dn/dlnM) * b(M,z) * M / ρ_m  dlnM = 1 at all z
# ============================================================

hmf  = np.array(get_cl.hmf_Mz_mat)   # (nz, nM)
bias = np.array(get_cl.bias_Mz_mat)  # (nz, nM)
rho_m = float(get_cl.rho_m_bar)      # mean comoving matter density

integrand = hmf * bias * M[None, :] / rho_m   # (nz, nM)
bias_integral = np.array([scipy_trap(integrand[jz], x=np.log(M))
                           for jz in range(len(z))])

fig, ax = pl.subplots(figsize=(8, 4))
ax.plot(z, bias_integral, 'b-o', lw=2, ms=4, label='computed')
ax.axhline(1.0, color='r', ls='--', lw=1.5, label='expected = 1')
ax.fill_between(z, 0.97, 1.03, alpha=0.15, color='green', label='±3% band')
ax.set_xlabel('z', fontsize=12)
ax.set_ylabel(r'$\int \frac{dn}{d\ln M}\, b(M,z)\, \frac{M}{\bar{\rho}_m}\, d\ln M$', fontsize=11)
ax.set_title('Tinker 2010 bias consistency (should equal 1 at all z)', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test1_2_bias_consistency.png', dpi=100, bbox_inches='tight')
pl.show()

max_dev = np.max(np.abs(bias_integral - 1.0))
print(f'Max deviation from 1: {max_dev:.4f}  ({100*max_dev:.2f}%)')
if max_dev < 0.05:
    print('PASS: bias consistency integral ≈ 1 at all z (within 5%)')
else:
    print(f'WARNING: deviation {100*max_dev:.1f}% — check HMF/bias normalisation or mass grid range')

In [ ]:
# ============================================================
#  TEST 1.3 — Baryon mass budget
#  fgas + f_star = Ω_b / Ω_m at every (z, M)
#  fgas >= 0 everywhere (gas fraction non-negative)
# ============================================================

fb    = get_cl.cosmo_params['Ob0'] / get_cl.cosmo_params['Om0']
fgas  = np.array(get_cl.fgas_mat)       # (nz, nM)
fstar = np.array(get_cl.fstar_tot_mat)  # (nz, nM)
budget_residual = np.abs(fgas + fstar - fb)

fig, axes = pl.subplots(1, 3, figsize=(16, 5))

im0 = axes[0].pcolormesh(np.log10(M), z, fgas, cmap='RdBu_r',
                          vmin=-0.01, vmax=fb)
pl.colorbar(im0, ax=axes[0], label=r'$f_{\rm gas}$')
axes[0].contour(np.log10(M), z, fgas, levels=[0.0], colors='red', linewidths=2)
axes[0].set_xlabel(r'$\log_{10}(M)$', fontsize=11)
axes[0].set_ylabel('z', fontsize=11)
axes[0].set_title('Gas fraction (red contour = 0)', fontsize=10)

for jz_i, label in [(0, f'z={z[0]:.2f}'), (len(z)//2, f'z={z[len(z)//2]:.2f}'),
                     (-1, f'z={z[-1]:.2f}')]:
    axes[1].plot(np.log10(M), fgas[jz_i] + fstar[jz_i], label=label)
axes[1].axhline(fb, color='k', ls='--', label=f'$\Omega_b/\Omega_m={fb:.3f}$')
axes[1].set_xlabel(r'$\log_{10}(M)$', fontsize=11)
axes[1].set_ylabel(r'$f_{\rm gas} + f_\star$', fontsize=11)
axes[1].set_title('Baryon budget (should = Ωb/Ωm)', fontsize=10)
axes[1].legend(fontsize=8)

im2 = axes[2].pcolormesh(np.log10(M), z, np.log10(budget_residual + 1e-12), cmap='viridis')
pl.colorbar(im2, ax=axes[2], label=r'$\log_{10}|f_{\rm gas}+f_\star - f_b|$')
axes[2].set_xlabel(r'$\log_{10}(M)$', fontsize=11)
axes[2].set_ylabel('z', fontsize=11)
axes[2].set_title('Budget residual', fontsize=10)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test1_3_baryon_budget.png', dpi=100, bbox_inches='tight')
pl.show()

neg_frac = np.sum(fgas < 0) / fgas.size
max_res  = budget_residual.max()
print(f'Ωb/Ωm target : {fb:.4f}')
print(f'Cells with fgas < 0 : {100*neg_frac:.2f}%')
print(f'Max budget residual : {max_res:.2e}')
if neg_frac < 0.02:
    print('PASS: fgas >= 0 in >98% of cells')
else:
    print(f'WARNING: {100*neg_frac:.1f}% of cells have fgas < 0 — '
          'stellar fraction exceeds baryon budget at high M')

In [ ]:
# ============================================================
#  TEST 2.1 — C_yy two-path consistency
#  get_Cl_tot(0,0,3,3) must equal manual trapz over cached_power_spectra[3,3]
# ============================================================
# Path A : get_Cl_tot(0,0,3,3)  — uses jit + trapz inside JAX
Cl_yy_A = np.array(get_cl.Cl_y_y_tot_mat)   # (nell,)

# Path B : manual numpy trapezoid over cached_power_spectra[3,3]
Pk_yy   = np.array(get_cl.cached_power_spectra[3, 3])   # (nell, nz_for_Cls)
z_fcls  = np.array(get_cl.z_array_for_Cls)
chi_f   = np.array(get_cl.chi_array_for_Cls)
dchi_f  = np.array(get_cl.dchi_dz_array_for_Cls)
Wy      = 1.0 / (1.0 + z_fcls)
prefac  = Wy / chi_f**2
fx      = prefac[None,:]**2 * chi_f[None,:]**2 * dchi_f[None,:] * Pk_yy  # (nell, nz)
Cl_yy_B = scipy_trap(fx, x=z_fcls, axis=1)   # (nell,)

rel_diff = np.abs(Cl_yy_A - Cl_yy_B) / np.abs(Cl_yy_A + 1e-40)
ell_arr  = np.array(l_eff)

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

ell_fac = ell_arr * (ell_arr + 1) / (2 * np.pi)
axes[0].loglog(ell_arr, ell_fac * Cl_yy_A, 'b-',  lw=2, label='Path A: get_Cl_tot(0,0,3,3)')
axes[0].loglog(ell_arr, ell_fac * Cl_yy_B, 'r--', lw=1.5, label='Path B: manual numpy trapz')
axes[0].set_xlabel(r'$\ell$', fontsize=12)
axes[0].set_ylabel(r'$\ell(\ell+1)C_\ell^{yy}/2\pi$', fontsize=12)
axes[0].set_title('Two-path consistency', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(ell_arr, 100 * rel_diff, 'k-', lw=2)
axes[1].axhline(0.1, color='r', ls='--', label='0.1% threshold')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel('Relative difference (%)', fontsize=12)
axes[1].set_title('|Path A − Path B| / Path A', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test2_1_twopaths.png', dpi=100, bbox_inches='tight')
pl.show()

max_rel = rel_diff.max()
print(f'Max relative difference: {100*max_rel:.4f}%')
if max_rel < 1e-4:
    print('PASS: two paths agree at < 0.01% — get_Cl_tot and cached_power_spectra[3,3] are consistent')
else:
    print(f'WARNING: relative difference is {100*max_rel:.2f}% — check trapz precision or JAX vs numpy')

In [ ]:
# ============================================================
#  TEST 2.2 — Pyy 3D decomposition + C_yy 1h/2h crossover
#
#  Expected:
#    3D: 2h > 1h at low k, 1h > 2h at high k
#    Angular: 2h dominates ell < 200, 1h dominates ell > 500
#    Crossover in range [100, 1000]
# ============================================================
kPk      = np.array(get_cl.kPk_array)
by_kz    = np.array(get_cl.by_kz_mat)    # (nk, nz)
plin_kz  = np.array(get_cl.plin_kz_mat)  # (nk, nz)
Pyy_tot  = np.array(get_cl.Pyy_tot_kz_mat)  # (nk, nz)
Pyy_2h   = by_kz * by_kz * plin_kz
Pyy_1h   = Pyy_tot - Pyy_2h

# ---- 3D decomposition at three redshifts ----
z_coarse  = np.array(get_cl.z_array)
z_targets = [0.1, 0.5, 1.0]

fig, axes = pl.subplots(1, 2, figsize=(13, 5))
colors = ['C0', 'C1', 'C2']
for i, z_t in enumerate(z_targets):
    jz = np.argmin(np.abs(z_coarse - z_t))
    axes[0].loglog(kPk, Pyy_1h[:, jz], colors[i]+'-',  lw=1.5, label=f'1h z={z_coarse[jz]:.2f}')
    axes[0].loglog(kPk, Pyy_2h[:, jz], colors[i]+'--', lw=1.5, label=f'2h z={z_coarse[jz]:.2f}')
axes[0].set_xlabel(r'$k$ [h/Mpc]', fontsize=12)
axes[0].set_ylabel(r'$P_{yy}(k, z)$', fontsize=12)
axes[0].set_title('3D P_yy: 1h (solid) vs 2h (dashed)', fontsize=11)
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(True, alpha=0.3)

# ---- Angular C_yy 1h/2h using get_Cl_y_y_1h from get_cov ----
# NOTE: this uses chi_array / z_array (coarser grid) — small grid difference from Cl_y_y_tot_mat
Cl_yy_1h_cov = np.array(vmap(get_cov_obj.get_Cl_y_y_1h)(jnp.arange(nell)))
Cl_yy_tot    = np.array(get_cl.Cl_y_y_tot_mat)
Cl_yy_2h_approx = Cl_yy_tot - Cl_yy_1h_cov

axes[1].loglog(ell_arr, ell_fac * Cl_yy_1h_cov,    'C0-',  lw=2, label='1h (from get_cov)')
axes[1].loglog(ell_arr, np.abs(ell_fac * Cl_yy_2h_approx), 'C1--', lw=2, label='2h (= tot − 1h)')
axes[1].loglog(ell_arr, ell_fac * Cl_yy_tot,        'k:',   lw=2, label='total')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel(r'$\ell(\ell+1)C_\ell^{yy}/2\pi$', fontsize=12)
axes[1].set_title('Angular C_yy: 1h vs 2h crossover', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test2_2_1h_2h_crossover.png', dpi=100, bbox_inches='tight')
pl.show()

# Check Pyy_1h is positive
neg_1h = np.sum(Pyy_1h < 0) / Pyy_1h.size
print(f'Pyy_1h negative fraction: {100*neg_1h:.3f}%')
if neg_1h < 0.001:
    print('PASS: Pyy_1h is positive everywhere')
else:
    print(f'WARNING: Pyy_1h has {100*neg_1h:.2f}% negative values')

# Find crossover ell
diff_sign = np.sign(Cl_yy_1h_cov - np.abs(Cl_yy_2h_approx))
cross_idx  = np.where(np.diff(diff_sign))[0]
if len(cross_idx) > 0:
    ell_cross = ell_arr[cross_idx[0]]
    print(f'1h = 2h crossover at ell ≈ {ell_cross:.0f}')
    if 100 < ell_cross < 1000:
        print('PASS: crossover in expected range [100, 1000]')
    else:
        print(f'WARNING: crossover ell={ell_cross:.0f} outside expected range [100, 1000]')
else:
    print('No clean crossover found — one term dominates at all ell')

In [ ]:
# ============================================================
#  TEST 2.3 — σ₈ power-law scaling
#  C_yy ∝ σ₈^n,  n ≈ 7–9  (ell-dependent)
#  SLOW: reinstantiates get_Cl 4 times
# ============================================================
sigma8_values = [0.74, 0.78, 0.81, 0.84, 0.88]  # ±~10% around fiducial
Cl_yy_s8 = {}
for s8 in sigma8_values:
    p = copy.deepcopy(sim_params_dict)
    p['cosmo']['sigma8'] = s8
    cl_tmp = get_Cl(p, halo_params_dict, analysis_dict, other_params_dict)
    Cl_yy_s8[s8] = np.array(cl_tmp.Cl_y_y_tot_mat)
    print(f'  σ8={s8:.2f} done', flush=True)

s8_arr = np.array(sigma8_values)
s8_fid = sim_params_dict['cosmo']['sigma8']

# Power-law index at a few reference ell bins
ell_ref_idx = [nell//5, nell//2, 4*nell//5]
fig, axes = pl.subplots(1, 2, figsize=(13, 5))

for jl in ell_ref_idx:
    Cl_vec = np.array([Cl_yy_s8[s8][jl] for s8 in sigma8_values])
    axes[0].loglog(s8_arr, Cl_vec / Cl_vec[sigma8_values.index(s8_fid)],
                   'o-', lw=2, label=fr'$\ell={ell_arr[jl]:.0f}$')

# Reference lines n=7, 8, 9
s8_plot = np.linspace(s8_arr[0], s8_arr[-1], 50)
for n, ls in zip([7, 8, 9], ['--', '-', ':']):
    axes[0].loglog(s8_plot, (s8_plot/s8_fid)**n, 'grey', ls=ls, lw=1, label=f'n={n}')
axes[0].set_xlabel(r'$\sigma_8$', fontsize=12)
axes[0].set_ylabel(r'$C_\ell^{yy}(\sigma_8) / C_\ell^{yy}(\sigma_8^{\rm fid})$', fontsize=11)
axes[0].set_title(r'$C_\ell^{yy}$ vs $\sigma_8$ (normalised to fiducial)', fontsize=11)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Log-slope d(log C_yy)/d(log σ₈) at each ell
log_s8  = np.log(s8_arr)
n_ell   = np.array([np.polyfit(log_s8,
                                np.log([Cl_yy_s8[s8][jl] for s8 in sigma8_values]), 1)[0]
                    for jl in range(nell)])
axes[1].semilogx(ell_arr, n_ell, 'b-', lw=2)
axes[1].axhspan(7, 9, alpha=0.2, color='green', label='expected n ∈ [7, 9]')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel(r'$d\log C_\ell^{yy} / d\log\sigma_8$', fontsize=12)
axes[1].set_title(r'Power-law index $n(\ell)$ — expected 7–9', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test2_3_sigma8_scaling.png', dpi=100, bbox_inches='tight')
pl.show()

n_med = np.median(n_ell)
print(f'Median power-law index n = {n_med:.2f}')
if 6.0 < n_med < 11.0:
    print('PASS: σ₈ scaling index in expected range [6, 11]')
else:
    print(f'WARNING: n={n_med:.2f} outside expected range — check P_yy normalisation')

In [ ]:
# ============================================================
#  TEST 2.4 — Redshift integration convergence
#  > 97% of C_yy power should come from z < 3
#  SLOW: reinstantiates get_Cl for each z_max
# ============================================================
zmax_values = [1.5, 2.0, 3.0, 4.0, 6.0]
Cl_yy_zmax = {}
for zm in zmax_values:
    a = copy.deepcopy(analysis_dict)
    a['zmax_for_Cls'] = zm
    cl_tmp = get_Cl(sim_params_dict, halo_params_dict, a, other_params_dict)
    Cl_yy_zmax[zm] = np.array(cl_tmp.Cl_y_y_tot_mat)
    print(f'  z_max={zm:.1f} done', flush=True)

Cl_ref = Cl_yy_zmax[6.0]  # reference: largest z_max

fig, axes = pl.subplots(1, 2, figsize=(13, 5))
colors = pl.cm.viridis(np.linspace(0.2, 0.9, len(zmax_values)))

for zm, c in zip(zmax_values, colors):
    axes[0].loglog(ell_arr, ell_fac * Cl_yy_zmax[zm], color=c, lw=1.8,
                   label=f'z_max={zm:.1f}')
axes[0].set_xlabel(r'$\ell$', fontsize=12)
axes[0].set_ylabel(r'$\ell(\ell+1)C_\ell^{yy}/2\pi$', fontsize=12)
axes[0].set_title('C_yy vs z_max integration limit', fontsize=11)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

for zm, c in zip(zmax_values[:-1], colors[:-1]):
    ratio = Cl_yy_zmax[zm] / np.clip(Cl_ref, 1e-40, None)
    axes[1].semilogx(ell_arr, 100 * ratio, color=c, lw=1.8, label=f'z_max={zm:.1f}')
axes[1].axhline(97, color='r', ls='--', label='97% threshold')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel('C_yy(z_max) / C_yy(z_max=6) [%]', fontsize=11)
axes[1].set_title('Fraction of total power vs z_max', fontsize=11)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test2_4_z_convergence.png', dpi=100, bbox_inches='tight')
pl.show()

frac_z3 = np.median(Cl_yy_zmax[3.0] / np.clip(Cl_ref, 1e-40, None))
print(f'Fraction of total power from z < 3.0: {100*frac_z3:.1f}%')
if frac_z3 > 0.97:
    print('PASS: >97% of C_yy power comes from z < 3')
else:
    print(f'NOTE: z < 3 contributes {100*frac_z3:.1f}% — extending z_max may help')

In [ ]:
# ============================================================
#  TEST 2.5 — Mass range convergence
#  Most C_yy power from M ~ few×10^14 M☉/h
#  Raising M_min from 1e11 to 1e13 should change C_yy < 10%
#  SLOW: reinstantiates get_Cl for each M_min
# ============================================================
lg10_Mmin_values = [11.0, 12.0, 13.0, 13.5]
Cl_yy_Mmin = {}
for lgM in lg10_Mmin_values:
    h = copy.deepcopy(halo_params_dict)
    h['lg10_Mmin'] = lgM
    cl_tmp = get_Cl(sim_params_dict, h, analysis_dict, other_params_dict)
    Cl_yy_Mmin[lgM] = np.array(cl_tmp.Cl_y_y_tot_mat)
    print(f'  lg10_Mmin={lgM:.1f} done', flush=True)

Cl_ref_M = Cl_yy_Mmin[11.0]  # reference: lowest M_min (most inclusive)

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

for lgM in lg10_Mmin_values:
    axes[0].loglog(ell_arr, ell_fac * Cl_yy_Mmin[lgM], lw=1.8,
                   label=f'lg10(Mmin)={lgM:.1f}')
axes[0].set_xlabel(r'$\ell$', fontsize=12)
axes[0].set_ylabel(r'$\ell(\ell+1)C_\ell^{yy}/2\pi$', fontsize=12)
axes[0].set_title('C_yy vs M_min cut', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

for lgM in lg10_Mmin_values[1:]:
    ratio = Cl_yy_Mmin[lgM] / np.clip(Cl_ref_M, 1e-40, None)
    axes[1].semilogx(ell_arr, 100 * ratio, lw=1.8, label=f'Mmin=10^{lgM:.1f}')
axes[1].axhline(90, color='r', ls='--', label='90% threshold')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel('C_yy(Mmin) / C_yy(Mmin=1e11) [%]', fontsize=11)
axes[1].set_title('Power fraction vs lower mass cut', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test2_5_mass_convergence.png', dpi=100, bbox_inches='tight')
pl.show()

frac_M13 = np.median(Cl_yy_Mmin[13.0] / np.clip(Cl_ref_M, 1e-40, None))
print(f'Fraction of power from M > 1e13 M☉/h: {100*frac_M13:.1f}%')
if frac_M13 > 0.85:
    print('PASS: >85% of C_yy power comes from M > 1e13 M☉/h')
else:
    print(f'NOTE: low-mass halos contribute {100*(1-frac_M13):.1f}% of C_yy — '
          'pressure normalisation may extend to low M')

In [ ]:
# ============================================================
#  TEST 3.1 — Gaussian covariance = Knox formula
#  diag(covG['yy_yy']) = 2 / [(2ℓ+1) fsky Δℓ] * (C_yy + N_yy)²
# ============================================================
covG = get_cov_obj.covG_dict['yy_yy']['bin_0_0_0_0']   # (nell, nell)
diag_G = np.diag(covG)

# Check off-diagonal is zero
off_diag_max = np.max(np.abs(covG - np.diag(diag_G)))
print(f'Max off-diagonal element of covG: {off_diag_max:.3e}')

# Manual Knox formula
Cl_yy_survey = np.array(get_cov_obj.Cl_result_dict['yy']['bin_0_0']['tot_plus_noise_ellsurvey'])
dl_survey    = np.array(analysis_dict['dl_array_survey'])
fsky_yy      = analysis_dict['fsky_yy']
knox_diag    = 2.0 / ((2 * ell_arr + 1) * fsky_yy * dl_survey) * Cl_yy_survey**2

rel_diff_knox = np.abs(diag_G - knox_diag) / np.abs(knox_diag + 1e-40)

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

axes[0].loglog(ell_arr, diag_G,   'b-',  lw=2,   label='covG diagonal')
axes[0].loglog(ell_arr, knox_diag, 'r--', lw=1.5, label='Knox formula')
axes[0].set_xlabel(r'$\ell$', fontsize=12)
axes[0].set_ylabel(r'$\sigma^2(C_\ell^{yy})$', fontsize=12)
axes[0].set_title('Gaussian covariance vs Knox formula', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(ell_arr, 100 * rel_diff_knox, 'k-', lw=2)
axes[1].axhline(0.1, color='r', ls='--', label='0.1% threshold')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel('Relative difference (%)', fontsize=12)
axes[1].set_title('|covG − Knox| / Knox', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test3_1_knox.png', dpi=100, bbox_inches='tight')
pl.show()

print(f'Max Knox relative difference: {100*rel_diff_knox.max():.4f}%')
if rel_diff_knox.max() < 1e-3:
    print('PASS: covG diagonal exactly matches Knox formula')
else:
    print(f'WARNING: Knox deviation {100*rel_diff_knox.max():.2f}%')

if off_diag_max < 1e-30:
    print('PASS: covG is diagonal (off-diagonal elements are zero)')
else:
    print(f'NOTE: max off-diagonal = {off_diag_max:.2e}')

In [ ]:
# ============================================================
#  TEST 3.2 — Gaussian covariance scales as 1/fsky
# ============================================================
fsky_new = 0.4   # 4× larger than fiducial 0.1

a2 = copy.deepcopy(analysis_dict)
a2['fsky_yy'] = fsky_new
cov2 = get_cov(sim_params_dict, halo_params_dict, a2, other_params_dict, Cl_obj=get_cl)

covG_new  = cov2.covG_dict['yy_yy']['bin_0_0_0_0']
diag_new  = np.diag(covG_new)
diag_orig = np.diag(get_cov_obj.covG_dict['yy_yy']['bin_0_0_0_0'])

expected_ratio = fsky_yy / fsky_new   # 0.1/0.4 = 0.25
actual_ratio   = diag_new / np.clip(diag_orig, 1e-40, None)

fig, ax = pl.subplots(figsize=(8, 4))
ax.semilogx(ell_arr, actual_ratio, 'b-', lw=2, label='covG(0.4) / covG(0.1)')
ax.axhline(expected_ratio, color='r', ls='--', lw=2, label=f'expected = fsky_old/fsky_new = {expected_ratio:.2f}')
ax.set_xlabel(r'$\ell$', fontsize=12)
ax.set_ylabel('Ratio', fontsize=12)
ax.set_title('covG scales as 1/fsky', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)
pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test3_2_fsky_scaling.png', dpi=100, bbox_inches='tight')
pl.show()

max_dev = np.max(np.abs(actual_ratio - expected_ratio))
print(f'Expected ratio: {expected_ratio:.4f}')
print(f'Max deviation from expected: {max_dev:.2e}')
if max_dev < 1e-10:
    print('PASS: covG scales exactly as 1/fsky')
else:
    print(f'WARNING: fsky scaling deviation {max_dev:.2e}')

In [ ]:
# ============================================================
#  TEST 3.3 — Positive definiteness of total covariance
# ============================================================
covtot = get_cov_obj.covtot_dict['yy_yy']['bin_0_0_0_0']   # (nell, nell)
covG   = get_cov_obj.covG_dict['yy_yy']['bin_0_0_0_0']
covNG  = get_cov_obj.covNG_dict['yy_yy']['bin_0_0_0_0']

eigvals_tot = np.linalg.eigvalsh(covtot)
eigvals_G   = np.linalg.eigvalsh(covG)
eigvals_NG  = np.linalg.eigvalsh(covNG)

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

axes[0].semilogy(eigvals_tot[::-1], 'b-', lw=2, label='covtot')
axes[0].semilogy(eigvals_G[::-1],   'g-', lw=1.5, alpha=0.8, label='covG')
axes[0].semilogy(np.abs(eigvals_NG[::-1]), 'r-', lw=1.5, alpha=0.8, label='|covNG|')
axes[0].axhline(0, color='k', ls='--', lw=0.8)
axes[0].set_xlabel('Eigenvalue index (sorted descending)', fontsize=11)
axes[0].set_ylabel('Eigenvalue', fontsize=11)
axes[0].set_title('Eigenvalue spectrum', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(np.log10(np.abs(covtot)), aspect='auto', cmap='RdBu_r')
pl.colorbar(im, ax=axes[1], label=r'$\log_{10}|\mathrm{Cov}|$')
axes[1].set_xlabel(r'$\ell$ index', fontsize=11)
axes[1].set_ylabel(r'$\ell$ index', fontsize=11)
axes[1].set_title('Total covariance matrix (log abs)', fontsize=11)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test3_3_positive_definite.png', dpi=100, bbox_inches='tight')
pl.show()

n_neg_tot = np.sum(eigvals_tot < 0)
min_eig   = eigvals_tot.min()
cond_num  = eigvals_tot.max() / np.abs(eigvals_tot[eigvals_tot > 0].min())
print(f'Negative eigenvalues (covtot): {n_neg_tot}')
print(f'Minimum eigenvalue           : {min_eig:.3e}')
print(f'Condition number             : {cond_num:.2e}')
if n_neg_tot == 0:
    print('PASS: total covariance matrix is positive definite')
else:
    print(f'WARNING: {n_neg_tot} negative eigenvalues — covNG may be too large')

In [ ]:
# ============================================================
#  TEST 4.1 + 4.2 — Non-Gaussian covariance
#  4.1: trispectrum diagonal >= 0
#  4.2: NG/G ratio vs ell — NG subdominant at low ell,
#       comparable at ell~3000
# ============================================================
covNG = get_cov_obj.covNG_dict['yy_yy']['bin_0_0_0_0']   # (nell, nell)
diag_NG = np.diag(covNG)
diag_G  = np.diag(get_cov_obj.covG_dict['yy_yy']['bin_0_0_0_0'])
ratio_NG_G = diag_NG / np.clip(diag_G, 1e-40, None)

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

# 4.1: trispectrum diagonal
n_neg_NG = np.sum(diag_NG < 0)
axes[0].loglog(ell_arr, diag_G,  'b-',  lw=2, label='Gaussian')
axes[0].loglog(ell_arr, np.abs(diag_NG), 'r-', lw=2, label='Non-Gaussian (|diag|)')
neg_mask = diag_NG < 0
if np.any(neg_mask):
    axes[0].scatter(ell_arr[neg_mask], np.abs(diag_NG[neg_mask]), c='red', marker='x',
                    zorder=5, s=50, label='negative NG (×)')
axes[0].set_xlabel(r'$\ell$', fontsize=12)
axes[0].set_ylabel(r'$\sigma^2(C_\ell^{yy})$', fontsize=12)
axes[0].set_title('Gaussian vs Non-Gaussian covariance diagonal', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# 4.2: NG/G ratio
axes[1].semilogx(ell_arr, ratio_NG_G, 'k-', lw=2)
axes[1].axhline(1.0, color='r', ls='--', label='NG = G (equal contribution)')
axes[1].axhline(0.1, color='g', ls=':', label='10% threshold')
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel(r'$\sigma^2_{\rm NG} / \sigma^2_G$', fontsize=12)
axes[1].set_title('Non-Gaussian / Gaussian ratio', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test4_1_2_NG_cov.png', dpi=100, bbox_inches='tight')
pl.show()

# Check off-diagonal structure of covNG (should be non-trivial, unlike covG)
covNG_offdiag_rms = np.sqrt(np.mean((covNG - np.diag(diag_NG))**2))
covG_offdiag_rms  = np.sqrt(np.mean((covG  - np.diag(diag_G ))**2))

print(f'Negative NG diagonal elements: {n_neg_NG}')
print(f'NG/G ratio range: [{ratio_NG_G.min():.3f}, {ratio_NG_G.max():.3f}]')
print(f'Off-diagonal RMS: covG = {covG_offdiag_rms:.2e},  covNG = {covNG_offdiag_rms:.2e}')

if n_neg_NG == 0:
    print('PASS: NG diagonal is non-negative')
else:
    print(f'WARNING: {n_neg_NG} negative NG diagonal elements')

if covNG_offdiag_rms > covG_offdiag_rms:
    print('PASS: covNG has significant off-diagonal structure (as expected for trispectrum)')
else:
    print('NOTE: covNG off-diagonal similar to covG — trispectrum may be dominated by diagonal')

In [ ]:
# ============================================================
#  TEST 4.3 — Upper mass cut reduces NG covariance
#  Masking halos M > 3e14 M☉/h reduces covNG by ~3–5×
#  while covG changes < 5%
#  SLOW: reinstantiates get_cov with modified Mmax
# ============================================================
lg10_Mmax_cut = 14.5   # mask halos above 3e14 M_sun/h

h_cut = copy.deepcopy(halo_params_dict)
h_cut['lg10_Mmax'] = lg10_Mmax_cut

print(f'Computing get_cov with lg10_Mmax={lg10_Mmax_cut} (slow)...', flush=True)
t0 = time.time()
cov_cut = get_cov(sim_params_dict, h_cut, analysis_dict, other_params_dict)
print(f'  done in {time.time()-t0:.1f}s', flush=True)

diag_NG_cut  = np.diag(cov_cut.covNG_dict['yy_yy']['bin_0_0_0_0'])
diag_G_cut   = np.diag(cov_cut.covG_dict ['yy_yy']['bin_0_0_0_0'])
diag_NG_full = np.diag(get_cov_obj.covNG_dict['yy_yy']['bin_0_0_0_0'])
diag_G_full  = np.diag(get_cov_obj.covG_dict ['yy_yy']['bin_0_0_0_0'])

ratio_NG = diag_NG_cut / np.clip(diag_NG_full, 1e-40, None)
ratio_G  = diag_G_cut  / np.clip(diag_G_full,  1e-40, None)

fig, axes = pl.subplots(1, 2, figsize=(13, 5))

axes[0].loglog(ell_arr, np.abs(diag_NG_full), 'b-',  lw=2, label=f'full Mmax')
axes[0].loglog(ell_arr, np.abs(diag_NG_cut),  'r--', lw=2, label=f'Mmax=10^{lg10_Mmax_cut}')
axes[0].set_xlabel(r'$\ell$', fontsize=12)
axes[0].set_ylabel(r'$\sigma^2_{\rm NG}$', fontsize=12)
axes[0].set_title('NG covariance: effect of upper mass cut', fontsize=11)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(ell_arr, ratio_NG, 'r-', lw=2, label='covNG(cut) / covNG(full)')
axes[1].semilogx(ell_arr, ratio_G,  'b-', lw=2, label='covG(cut) / covG(full)')
axes[1].axhline(1.0, color='grey', ls='--', lw=1)
axes[1].set_xlabel(r'$\ell$', fontsize=12)
axes[1].set_ylabel('Ratio (cut / full)', fontsize=12)
axes[1].set_title(f'Effect of masking M > 10^{lg10_Mmax_cut} M☉/h', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pl.tight_layout()
pl.savefig(OUTPUT_DIR / 'tests/test4_3_mass_cut_NG.png', dpi=100, bbox_inches='tight')
pl.show()

med_NG_ratio = np.median(ratio_NG)
med_G_ratio  = np.median(ratio_G)
print(f'Median covNG reduction from mass cut: {100*(1-med_NG_ratio):.1f}%  (ratio={med_NG_ratio:.3f})')
print(f'Median covG  change  from mass cut  : {100*(1-med_G_ratio):.1f}%  (ratio={med_G_ratio:.3f})')
if med_NG_ratio < 0.8:
    print('PASS: mass cut significantly reduces NG covariance (>20%)')
else:
    print('NOTE: mass cut has small effect — most NG power may come from intermediate mass halos')
if np.abs(1 - med_G_ratio) < 0.1:
    print('PASS: mass cut has minimal effect on Gaussian covariance (<10%)')
else:
    print(f'WARNING: mass cut changes covG by {100*np.abs(1-med_G_ratio):.1f}%')